# Epoch/Round Tuning Analysis

This notebook reads the grouped `epoch_round_tuning/tuning-*` runs, summarizes per-round validation MMLU, compares the selected epoch/round pairs across linear FLoRA, nonlinear FLoRA, and nonlinear FFA, and renders Plotly heatmaps for the epoch x round grid.

In [8]:
import os
from pathlib import Path

from IPython.display import display
import plotly.express as px

from utils.tuning_analysis import (
    build_extension_requests,
    build_llama_confirmation_requests,
    build_repeat_requests,
    compare_selected_to_paper,
    load_tuning_results,
    make_tuning_heatmap,
    make_tuning_round_curves,
    select_plateaus,
    summarize_tuning_results,
    write_manifest,
)

CHROME_PATH = Path.home() / '.cache' / 'plotly-chrome' / 'chrome-linux64' / 'chrome'
if CHROME_PATH.exists():
    os.environ.setdefault('BROWSER_PATH', str(CHROME_PATH))

BASE_DIR = Path('.')
TUNING_RUN_ROOTS = [
    # BASE_DIR / 'epoch_round_tuning',
    BASE_DIR / 'epoch_round_tuning_r10',
    BASE_DIR / 'epoch_round_tuning_dolly_dirichlet_r10',
    BASE_DIR / 'exp2-tinyllama-nonlinear-wiz',
    BASE_DIR / 'exp2-tinyllama-nonlinear-3round-heter-wiz',
    BASE_DIR / 'epoch_round_tuning_wiz_stratified_r10',
    # BASE_DIR / 'exp2-llama-nonlinear-1round-homo-wiz',
    # BASE_DIR / 'exp2-llama-nonlinear-1round-heter-wiz',
]
OUTPUT_DIR = BASE_DIR / 'tuning_analysis_outputs'
FIGURE_DIR = OUTPUT_DIR / 'epoch_round_heatmaps'
ROUND_CURVE_DIR = BASE_DIR / 'plots_epoch_round_tuning'

OUTPUT_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)
ROUND_CURVE_DIR.mkdir(exist_ok=True)


## Load And Select

In [9]:
tuning_scores = load_tuning_results(TUNING_RUN_ROOTS)
tuning_summary = summarize_tuning_results(tuning_scores)
tuning_plateaus, tuning_selected = select_plateaus(tuning_summary)

run_count = tuning_scores['Run dir'].nunique() if not tuning_scores.empty else 0
case_count = tuning_selected.shape[0] if not tuning_selected.empty else 0
print(f'Loaded {len(tuning_scores)} per-round scores from {run_count} runs under {TUNING_RUN_ROOTS}.')
print(f'Selected {case_count} method/dataset/model/setting configurations.')

display(tuning_selected.round(2))
display(tuning_plateaus.round(2))

Loaded 586 per-round scores from 60 runs under [PosixPath('epoch_round_tuning_r10'), PosixPath('epoch_round_tuning_dolly_dirichlet_r10'), PosixPath('exp2-tinyllama-nonlinear-wiz'), PosixPath('exp2-tinyllama-nonlinear-3round-heter-wiz'), PosixPath('epoch_round_tuning_wiz_stratified_r10')].
Selected 18 method/dataset/model/setting configurations.


,Method key,Method,Dataset,Dataset label,Model key,Model,Setting key,Setting,Selected epochs,Selected round,Selected accuracy,Selected std,Selected seed count,Selected seeds,Selected compute cost,Best epochs,Best round,Best accuracy,Accuracy gap to best,Low-cost tie count
0,ffa,Nonlinear FFA,dolly,Dolly,tinyllama,TinyLlama,heter,Heter,3,4,28.52,0.0,1,0,12,5,9,29.05,0.52,1
1,ffa,Nonlinear FFA,dolly,Dolly,tinyllama,TinyLlama,homo,Homo,2,3,29.90,0.0,1,0,6,5,7,30.87,0.97,1
4,ffa,Nonlinear FFA,dolly_stratified,Dolly stratified,tinyllama,TinyLlama,heter,Heter,3,5,29.62,0.0,1,0,15,3,5,29.62,0.00,1
5,ffa,Nonlinear FFA,dolly_stratified,Dolly stratified,tinyllama,TinyLlama,homo,Homo,3,7,30.62,0.0,1,0,21,3,10,31.26,0.65,1
8,ffa,Nonlinear FFA,wiz,Wizard,tinyllama,TinyLlama,heter,Heter,1,6,33.19,0.0,1,0,6,1,10,33.63,0.44,1
9,ffa,Nonlinear FFA,wiz,Wizard,tinyllama,TinyLlama,homo,Homo,2,1,30.69,0.0,1,0,2,5,4,31.55,0.86,2
14,ffa,Nonlinear FFA,wiz_stratified,Wizard stratified,tinyllama,TinyLlama,heter,Heter,1,5,32.34,0.0,1,0,5,1,8,33.07,0.73,1
15,ffa,Nonlinear FFA,wiz_stratified,Wizard stratified,tinyllama,TinyLlama,homo,Homo,1,2,31.25,0.0,1,0,2,2,5,31.84,0.59,1
2,flora,Linear FLoRA,dolly,Dolly,tinyllama,TinyLlama,heter,Heter,2,1,29.38,0.0,1,0,2,2,5,29.96,0.58,1
3,flora,Linear FLoRA,dolly,Dolly,tinyllama,TinyLlama,homo,Homo,3,2,31.03,0.0,1,0,6,3,2,31.03,0.00,1


,Method key,Method,Dataset,Dataset label,Model key,Model,Setting key,Setting,Local epochs,Plateau round,Plateau accuracy,Best round,Best accuracy,Max round observed,Declined after best
0,ffa,Nonlinear FFA,dolly,Dolly,tinyllama,TinyLlama,heter,Heter,1,8,26.33,10,27.24,10,False
1,ffa,Nonlinear FFA,dolly,Dolly,tinyllama,TinyLlama,heter,Heter,2,7,27.52,9,28.38,10,False
2,ffa,Nonlinear FFA,dolly,Dolly,tinyllama,TinyLlama,heter,Heter,3,4,28.52,6,28.99,10,True
3,ffa,Nonlinear FFA,dolly,Dolly,tinyllama,TinyLlama,heter,Heter,5,4,28.88,9,29.05,10,True
4,ffa,Nonlinear FFA,dolly,Dolly,tinyllama,TinyLlama,homo,Homo,1,6,29.29,6,29.29,10,True
5,ffa,Nonlinear FFA,dolly,Dolly,tinyllama,TinyLlama,homo,Homo,2,3,29.90,9,30.31,10,False
6,ffa,Nonlinear FFA,dolly,Dolly,tinyllama,TinyLlama,homo,Homo,3,4,29.32,7,29.56,10,False
7,ffa,Nonlinear FFA,dolly,Dolly,tinyllama,TinyLlama,homo,Homo,5,5,30.35,7,30.87,10,True
16,ffa,Nonlinear FFA,dolly_stratified,Dolly stratified,tinyllama,TinyLlama,heter,Heter,1,6,25.59,9,26.40,10,False
17,ffa,Nonlinear FFA,dolly_stratified,Dolly stratified,tinyllama,TinyLlama,heter,Heter,2,5,27.69,9,28.42,10,True


In [10]:
tuning_scores.to_csv(OUTPUT_DIR / 'tuning_scores.csv', index=False)
tuning_summary.to_csv(OUTPUT_DIR / 'tuning_summary.csv', index=False)
tuning_plateaus.to_csv(OUTPUT_DIR / 'tuning_plateaus.csv', index=False)
tuning_selected.to_csv(OUTPUT_DIR / 'tuning_selected.csv', index=False)
print(f'Wrote CSV summaries to {OUTPUT_DIR}.')

Wrote CSV summaries to tuning_analysis_outputs.


## Method Alignment

In [11]:
if tuning_selected.empty:
    print('No selected configurations yet.')
else:
    index_columns = ['Dataset label', 'Model', 'Setting']
    alignment = tuning_selected.pivot(
        index=index_columns,
        columns='Method',
        values=['Selected epochs', 'Selected round', 'Selected accuracy', 'Best epochs', 'Best round', 'Best accuracy'],
    )
    display(alignment.round(2))

    winners = (
        tuning_selected.sort_values('Selected accuracy', ascending=False)
        .groupby(index_columns, as_index=False)
        .first()[index_columns + ['Method', 'Selected epochs', 'Selected round', 'Selected accuracy', 'Best accuracy']]
    )
    display(winners.round(2))

Selected epochs                \
Method                                 Linear FLoRA Nonlinear FFA   
Dataset label     Model     Setting                                 
Dolly             TinyLlama Heter               2.0           3.0   
                            Homo                3.0           2.0   
Dolly stratified  TinyLlama Heter               5.0           3.0   
                            Homo                1.0           3.0   
Wizard            TinyLlama Heter               3.0           1.0   
                            Homo                1.0           2.0   
Wizard stratified TinyLlama Heter               3.0           1.0   
                            Homo                1.0           1.0   

                                                    Selected round  \
Method                              Nonlinear FLoRA   Linear FLoRA   
Dataset label     Model     Setting                                  
Dolly             TinyLlama Heter               NaN            1.0   
                            Homo                NaN            2.0   
Dolly stratified  TinyLlama Heter               NaN            1.0   
                            Homo                NaN            5.0   
Wizard            TinyLlama Heter               1.0            1.0   
                            Homo                1.0            1.0   
Wizard stratified TinyLlama Heter               NaN            1.0   
                            Homo                NaN            1.0   

                                                                   \
Method                              Nonlinear FFA Nonlinear FLoRA   
Dataset label     Model     Setting                                 
Dolly             TinyLlama Heter             4.0             NaN   
                            Homo              3.0             NaN   
Dolly stratified  TinyLlama Heter             5.0             NaN   
                            Homo              7.0             NaN   
Wizard            TinyLlama Heter             6.0             1.0   
                            Homo              1.0             1.0   
Wizard stratified TinyLlama Heter             5.0             NaN   
                            Homo              2.0             NaN   

                                    Selected accuracy                \
Method                                   Linear FLoRA Nonlinear FFA   
Dataset label     Model     Setting                                   
Dolly             TinyLlama Heter               29.38         28.52   
                            Homo                31.03         29.90   
Dolly stratified  TinyLlama Heter               28.34         29.62   
                            Homo                28.63         30.62   
Wizard            TinyLlama Heter               46.22         33.19   
                            Homo                45.10         30.69   
Wizard stratified TinyLlama Heter               45.74         32.34   
                            Homo                46.74         31.25   

                                                     Best epochs  \
Method                              Nonlinear FLoRA Linear FLoRA   
Dataset label     Model     Setting                                
Dolly             TinyLlama Heter               NaN          2.0   
                            Homo                NaN          3.0   
Dolly stratified  TinyLlama Heter               NaN          2.0   
                            Homo                NaN          5.0   
Wizard            TinyLlama Heter             35.94          3.0   
                            Homo              36.87          1.0   
Wizard stratified TinyLlama Heter               NaN          3.0   
                            Homo                NaN          1.0   

                                                                   \
Method                              Nonlinear FFA Nonlinear FLoRA   
Dataset label     Model     Setting                                 
Dolly  

,Dataset label,Model,Setting,Method,Selected epochs,Selected round,Selected accuracy,Best accuracy
0,Dolly,TinyLlama,Heter,Linear FLoRA,2,1,29.38,29.96
1,Dolly,TinyLlama,Homo,Linear FLoRA,3,2,31.03,31.03
2,Dolly stratified,TinyLlama,Heter,Nonlinear FFA,3,5,29.62,29.62
3,Dolly stratified,TinyLlama,Homo,Nonlinear FFA,3,7,30.62,31.26
4,Wizard,TinyLlama,Heter,Linear FLoRA,3,1,46.22,46.22
5,Wizard,TinyLlama,Homo,Linear FLoRA,1,1,45.10,45.10
6,Wizard stratified,TinyLlama,Heter,Linear FLoRA,3,1,45.74,45.74
7,Wizard stratified,TinyLlama,Homo,Linear FLoRA,1,1,46.74,46.74


## Paper Baseline Comparison

The Wizard and Dolly paper baselines are taken from `flora_results_analysis.ipynb`. Paper results are plotted only at their reported communication round, not as horizontal lines across all rounds. Dolly tuning here uses the stratified split, so the Dolly paper point is contextual rather than an exact split-matched baseline.

In [12]:
# paper_comparison = compare_selected_to_paper(tuning_selected)
# if paper_comparison.empty:
#     print('No paper baselines matched the selected tuning rows.')
# else:
#     paper_comparison.to_csv(OUTPUT_DIR / 'tuning_vs_paper.csv', index=False)
#     display(paper_comparison.round(2))


## Round Curves

The fixed-epoch method comparison plots answer the direct question: at local epoch 1, 2, 3, or 5, which method has the better round curve? The all-curve overview plots stay separate so the main comparison stays readable. I would avoid overlaying Dolly and Dolly stratified in this same view because it doubles the curve count and makes the method comparison harder to read; a split-focused view should be faceted by split if we want to inspect that question.


In [13]:
METHOD_ORDER = ['Linear FLoRA', 'Nonlinear FLoRA', 'Nonlinear FFA']
SETTING_ORDER = ['Homo', 'Heter']


def _slug(value):
    return str(value).lower().replace(' ', '_').replace('-', '_').replace('/', '_')


def _sorted_unique_ints(values):
    return sorted({int(value) for value in values})


def _category_orders(data):
    return {
        'Method': [method for method in METHOD_ORDER if method in set(data['Method'])],
        'Setting': [setting for setting in SETTING_ORDER if setting in set(data['Setting'])],
        'Local epochs': [str(epoch) for epoch in _sorted_unique_ints(data['Local epochs'])],
    }


def _format_facet_labels(fig):
    fig.for_each_annotation(lambda annotation: annotation.update(text=annotation.text.split('=')[-1]))


def _save_plot(fig, output_stem, png_warnings):
    output_stem.parent.mkdir(parents=True, exist_ok=True)
    png_path = output_stem.with_suffix('.png')
    try:
        fig.write_image(png_path, scale=2)
    except Exception as exc:
        png_path = None
        message = str(exc).strip() or type(exc).__name__
        png_warnings.append(message.splitlines()[0])
    return png_path


def make_epoch_method_comparison(summary, dataset, epoch):
    data = summary[
        (summary['Dataset'] == dataset)
        & (summary['Local epochs'].astype(int) == int(epoch))
    ].copy()
    if data.empty:
        return None

    data = data.sort_values(['Model', 'Setting', 'Method', 'Round'])
    dataset_label = data['Dataset label'].iloc[0]
    model_label = ', '.join(sorted(data['Model'].unique()))
    facet_row = 'Model' if data['Model key'].nunique() > 1 else None

    fig = px.line(
        data,
        x='Round',
        y='Mean accuracy',
        color='Method',
        facet_col='Setting',
        facet_row=facet_row,
        markers=True,
        template='plotly_white',
        title=f'{dataset_label} {model_label} epoch {epoch}: method comparison',
        labels={'Mean accuracy': 'MMLU accuracy (%)'},
        category_orders=_category_orders(data),
        hover_data={
            'Local epochs': True,
            'Std accuracy': ':.2f',
            'Seed count': True,
        },
    )
    fig.update_layout(width=1000, height=400, legend_title_text='')
    fig.update_xaxes(dtick=1, showgrid=True, gridcolor='#d9d9d9', gridwidth=1)
    fig.update_yaxes(showgrid=True, gridcolor='#d9d9d9', gridwidth=1)
    _format_facet_labels(fig)
    return fig


def make_dataset_all_curves(summary, dataset):
    data = summary[summary['Dataset'] == dataset].copy()
    if data.empty:
        return None

    data['Local epochs'] = data['Local epochs'].astype(int).astype(str)
    data = data.sort_values(['Model', 'Setting', 'Method', 'Local epochs', 'Round'])
    dataset_label = data['Dataset label'].iloc[0]
    facet_row = 'Model' if data['Model key'].nunique() > 1 else None

    fig = px.line(
        data,
        x='Round',
        y='Mean accuracy',
        color='Local epochs',
        line_dash='Method',
        facet_col='Setting',
        facet_row=facet_row,
        markers=True,
        template='plotly_white',
        title=f'{dataset_label}: all tuning round curves',
        labels={'Mean accuracy': 'MMLU accuracy (%)', 'Local epochs': 'Epochs'},
        category_orders=_category_orders(data),
        hover_data={
            'Std accuracy': ':.2f',
            'Seed count': True,
        },
    )
    fig.update_layout(width=1000, height=400, legend_title_text='')
    fig.update_xaxes(dtick=1, showgrid=True, gridcolor='#d9d9d9', gridwidth=1)
    fig.update_yaxes(showgrid=True, gridcolor='#d9d9d9', gridwidth=1)
    _format_facet_labels(fig)
    return fig


if tuning_summary.empty:
    print('No tuning summary to plot.')
else:
    png_warnings = []
    saved_png = []
    datasets = tuning_summary[['Dataset', 'Dataset label']].drop_duplicates().sort_values('Dataset label')
    epochs = _sorted_unique_ints(tuning_summary['Local epochs'])

    comparison_root = ROUND_CURVE_DIR / 'fixed_epoch_method_comparisons'
    overview_root = ROUND_CURVE_DIR / 'all_curves'

    print('Fixed-epoch method comparisons')
    for _, dataset_row in datasets.iterrows():
        dataset = dataset_row['Dataset']
        dataset_label = dataset_row['Dataset label']
        dataset_dir = comparison_root / _slug(dataset_label)
        for epoch in epochs:
            fig = make_epoch_method_comparison(tuning_summary, dataset, epoch)
            if fig is None:
                print(f'No data for {dataset_label} epoch {epoch}; skipping.')
                continue
            png_path = _save_plot(fig, dataset_dir / f'epoch_{epoch}', png_warnings)
            if png_path is not None:
                saved_png.append(png_path)
            print(f'{dataset_label} epoch {epoch}')
            fig.show()

    print('All-curve dataset overviews')
    for _, dataset_row in datasets.iterrows():
        dataset = dataset_row['Dataset']
        dataset_label = dataset_row['Dataset label']
        fig = make_dataset_all_curves(tuning_summary, dataset)
        if fig is None:
            continue
        png_path = _save_plot(fig, overview_root / _slug(dataset_label), png_warnings)
        if png_path is not None:
            saved_png.append(png_path)
        print(dataset_label)
        fig.show()

    if saved_png:
        print(f'Wrote {len(saved_png)} PNG files under {ROUND_CURVE_DIR}.')
    if png_warnings:
        print('PNG export skipped or incomplete. Install/enable kaleido for static PNG export.')
        print(f'First PNG export warning: {png_warnings[0]}')


Fixed-epoch method comparisons


Dolly epoch 1


Dolly epoch 2


Dolly epoch 3


Dolly epoch 5


...

All-curve dataset overviews
Dolly


Dolly stratified


Wizard


Wizard stratified


Wrote 20 PNG files under plots_epoch_round_tuning.


## Epoch x Round Heatmaps

In [14]:
# if tuning_summary.empty:
#     print('No tuning summary to plot.')
# else:
#     heatmap_cases = (
#         tuning_summary[['Method key', 'Dataset', 'Model key', 'Setting key']]
#         .drop_duplicates()
#         .sort_values(['Dataset', 'Model key', 'Setting key', 'Method key'])
#     )
#     for _, row in heatmap_cases.iterrows():
#         fig = make_tuning_heatmap(
#             tuning_summary,
#             method=row['Method key'],
#             dataset=row['Dataset'],
#             model=row['Model key'],
#             setting=row['Setting key'],
#         )
#         if fig is None:
#             continue
#         png_name = f"heatmap_{row['Method key']}_{row['Dataset']}_{row['Model key']}_{row['Setting key']}.png"
#         fig.write_image(FIGURE_DIR / png_name, scale=2)
#         fig.show()
#     print(f'Wrote heatmap PNG files to {FIGURE_DIR}.')
